In [ ]:
# Práctica: Cláusula COLLATION en SQL Server

## 🎯 Objetivo de Aprendizaje
Comprender cómo la cláusula `COLLATION` afecta el **ordenamiento** y **comparación** de caracteres en SQL Server, especialmente con caracteres especiales del español y otros idiomas.

## 📚 Conceptos Clave

**COLLATION** define las reglas para:
- **Ordenamiento** de datos (ORDER BY)
- **Comparación** de cadenas (WHERE, JOIN, =)
- **Sensibilidad** a mayúsculas, acentos y caracteres especiales

### Sufijos Comunes:
- **CI** = Case Insensitive (no distingue mayúsculas/minúsculas)
- **CS** = Case Sensitive (distingue mayúsculas/minúsculas)  
- **AI** = Accent Insensitive (no distingue acentos)
- **AS** = Accent Sensitive (distingue acentos)
- **BIN** = Binary (ordenamiento por código Unicode)

---

## 🔬 Duración estimada: 30-40 minutos

## Paso 1: Crear Tablas con Diferentes COLLATIONS

Vamos a crear 3 tablas idénticas pero con diferentes configuraciones de COLLATION para ver cómo afecta el ordenamiento.

In [ ]:
-- Limpiar tablas existentes
IF OBJECT_ID('dbo.Productos_Espanol', 'U') IS NOT NULL DROP TABLE dbo.Productos_Espanol;
IF OBJECT_ID('dbo.Productos_Ingles', 'U') IS NOT NULL DROP TABLE dbo.Productos_Ingles;
IF OBJECT_ID('dbo.Productos_Binario', 'U') IS NOT NULL DROP TABLE dbo.Productos_Binario;
GO

-- TABLA 1: COLLATION ESPAÑOL (Traditional Sort)
-- Ordena CH y LL como letras independientes
CREATE TABLE Productos_Espanol (
    ID INT IDENTITY(1,1) PRIMARY KEY,
    Nombre NVARCHAR(50) COLLATE Modern_Spanish_CI_AS
);

-- TABLA 2: COLLATION INGLÉS (Dictionary Order)
-- CH y LL se ordenan como C+H y L+L
CREATE TABLE Productos_Ingles (
    ID INT IDENTITY(1,1) PRIMARY KEY,
    Nombre NVARCHAR(50) COLLATE SQL_Latin1_General_CP1_CI_AS
);

-- TABLA 3: COLLATION BINARIA (Binary Sort)
-- Ordenamiento por valor numérico Unicode
CREATE TABLE Productos_Binario (
    ID INT IDENTITY(1,1) PRIMARY KEY,
    Nombre NVARCHAR(50) COLLATE Latin1_General_BIN
);
GO

## Paso 2: Insertar Datos de Prueba

Insertaremos 10 palabras con caracteres especiales:
- **ñ** (español): Año, Niño, Ñandú
- **ch** (español): Chocolate
- **ll** (español): Llama
- **Acentos**: Búho, Café
- **Otros idiomas**: Ångström (sueco), Açaí (portugués)

In [ ]:
-- Insertar datos en las 3 tablas
INSERT INTO Productos_Espanol (Nombre) VALUES
    (N'Año'),           -- ñ (español)
    (N'Búho'),          -- Acento
    (N'Chocolate'),     -- ch (español)
    (N'Llama'),         -- ll (español)
    (N'Niño'),          -- ñ en medio
    (N'Ñandú'),         -- Ñ mayúscula inicial
    (N'Café'),          -- Acento francés
    (N'Ångström'),      -- Å (sueco/nórdico)
    (N'Zorro'),         -- Z normal
    (N'Açaí');          -- ç (portugués)

INSERT INTO Productos_Ingles (Nombre) VALUES
    (N'Año'), (N'Búho'), (N'Chocolate'), (N'Llama'), (N'Niño'),
    (N'Ñandú'), (N'Café'), (N'Ångström'), (N'Zorro'), (N'Açaí');

INSERT INTO Productos_Binario (Nombre) VALUES
    (N'Año'), (N'Búho'), (N'Chocolate'), (N'Llama'), (N'Niño'),
    (N'Ñandú'), (N'Café'), (N'Ångström'), (N'Zorro'), (N'Açaí');

GO

-- Verificar inserción
SELECT '✓ Datos insertados correctamente' AS Resultado,
       (SELECT COUNT(*) FROM Productos_Espanol) AS Total_Productos;

## Paso 3: Comparar Ordenamiento por COLLATION

Ahora vamos a consultar las 3 tablas con ORDER BY para observar cómo cada COLLATION ordena los mismos datos de manera diferente.

In [ ]:
-- CONSULTA 1: COLLATION ESPAÑOL
-- Observa cómo CH y LL se ordenan como letras independientes
PRINT '═══════════════════════════════════════════════════';
PRINT 'COLLATION ESPAÑOL (Modern_Spanish_CI_AS)';
PRINT '═══════════════════════════════════════════════════';
SELECT ID, Nombre 
FROM Productos_Espanol 
ORDER BY Nombre;
GO

In [ ]:
-- CONSULTA 2: COLLATION INGLÉS
-- Observa cómo CH y LL se tratan como dos letras separadas (C+H, L+L)
PRINT '═══════════════════════════════════════════════════';
PRINT 'COLLATION INGLÉS (SQL_Latin1_General_CP1_CI_AS)';
PRINT '═══════════════════════════════════════════════════';
SELECT ID, Nombre 
FROM Productos_Ingles 
ORDER BY Nombre;
GO

In [ ]:
-- CONSULTA 3: COLLATION BINARIA
-- Observa el ordenamiento por valor numérico Unicode
PRINT '═══════════════════════════════════════════════════';
PRINT 'COLLATION BINARIA (Latin1_General_BIN)';
PRINT '═══════════════════════════════════════════════════';
SELECT ID, Nombre 
FROM Productos_Binario 
ORDER BY Nombre;
GO

## 📊 Resultados Esperados

### COLLATION ESPAÑOL (Modern_Spanish_CI_AS)
- **CH** se ordena después de todas las palabras que empiezan con C
- **LL** se ordena después de todas las palabras que empiezan con L
- **Ñ** se ordena después de N

### COLLATION INGLÉS (SQL_Latin1_General_CP1_CI_AS)
- **CH** se trata como C+H (orden alfabético normal)
- **LL** se trata como L+L (orden alfabético normal)
- **Caracteres con acentos** se ordenan junto a sus equivalentes sin acento

### COLLATION BINARIA (Latin1_General_BIN)
- Ordenamiento **estricto** por código numérico Unicode
- **Case Sensitive**: distingue mayúsculas de minúsculas
- Caracteres especiales pueden aparecer al final

## Paso 4: Demostración de Case Sensitivity

Veamos cómo COLLATION afecta la búsqueda de datos.

In [ ]:
-- BÚSQUEDA INSENSIBLE A MAYÚSCULAS (CI = Case Insensitive)
PRINT '═══════════════════════════════════════════════════';
PRINT 'Buscando "niño" (minúsculas) en tabla ESPAÑOL (CI)';
PRINT '═══════════════════════════════════════════════════';

SELECT Nombre, 'Encontrado (CI ignora mayúsculas)' AS Resultado
FROM Productos_Espanol 
WHERE Nombre = 'niño';  -- Encuentra "Niño" aunque busquemos "niño"

GO

In [ ]:
-- BÚSQUEDA SENSIBLE A MAYÚSCULAS (BIN = Case Sensitive)
PRINT '═══════════════════════════════════════════════════';
PRINT 'Buscando "niño" (minúsculas) en tabla BINARIA (CS)';
PRINT '═══════════════════════════════════════════════════';

SELECT Nombre, 'NO encontrado (BIN es case sensitive)' AS Resultado
FROM Productos_Binario 
WHERE Nombre = 'niño';  -- NO encuentra nada (está almacenado como "Niño")

PRINT '';
PRINT 'Buscando "Niño" (con mayúscula) en tabla BINARIA';

SELECT Nombre, 'Encontrado (coincide exactamente)' AS Resultado
FROM Productos_Binario 
WHERE Nombre = 'Niño';  -- SÍ encuentra (coincide exactamente)

GO

## Paso 5: Cambiar COLLATION en Tiempo de Consulta

Puedes forzar un COLLATION específico en una consulta individual sin cambiar la tabla.

In [ ]:
-- Forzar ordenamiento ESPAÑOL en la tabla INGLÉS
PRINT '═══════════════════════════════════════════════════';
PRINT 'Tabla INGLÉS ordenada CON collation español';
PRINT '═══════════════════════════════════════════════════';

SELECT ID, Nombre 
FROM Productos_Ingles 
ORDER BY Nombre COLLATE Modern_Spanish_CI_AS;

GO

## 🎓 Ejercicios para Estudiantes

### Ejercicio 1: Experimentar con más datos
Agrega 5 palabras más a las tablas con los siguientes caracteres:
- `ü` (alemán): Über
- `ø` (noruego): Søren  
- `ß` (alemán): Straße
- Más palabras con **ch** y **ll**

### Ejercicio 2: Crear tu propia COLLATION
Crea una nueva tabla con `COLLATE SQL_Latin1_General_CP1_CS_AS` (Case Sensitive) y compara los resultados.

### Ejercicio 3: Investigar
Ejecuta este comando para ver la COLLATION de tu servidor:
```sql
SELECT SERVERPROPERTY('Collation') AS Collation_Servidor;
```

---

## 💡 Conclusiones Clave

1. **COLLATION afecta el ordenamiento**: CH y LL se ordenan diferente en español vs inglés
2. **Case Sensitivity importa**: BIN distingue mayúsculas, CI no
3. **Puedes cambiar COLLATION**: A nivel de servidor, base de datos, tabla, columna o consulta
4. **Considera el idioma**: Usa `Modern_Spanish_CI_AS` para aplicaciones en español

---

## 📝 Entregable Sugerido

Capturas de pantalla mostrando:
1. Resultados de las 3 consultas de ordenamiento
2. Diferencias en búsquedas case-sensitive vs case-insensitive
3. Tus propios experimentos con nuevos datos